1、导包

In [1]:
#读取和操作表格数据
import pandas as pd
#导入标签编码器，用于把类别型文本转换为数值
from sklearn.preprocessing import LabelEncoder
#用于自定义数据集
from torch.utils.data import Dataset
#导入随机切分工具，用于把数据集按比例随机分成训练集，验证集
from torch.utils.data import random_split


2、准备数据

In [2]:

# 定义数据集
class CSVDataset(Dataset):
    # 导入数据集
    def __init__(self, path):
        # 导入传入路径的数据集为 Pandas DataFrame 格式
        df = pd.read_csv(path,header=None)
        # 设置神经网络的输入与输出
        self.X = df.values[:, :-1]  # 根据你的数据集定义输入属性
        self.y = df.values[:, -1]  # 根据你的数据集定义输出属性
        # 确保输入的数据是浮点型
        self.X = self.X.astype('float32')
        # 使用浮点型标签编码原输出
        self.y = LabelEncoder().fit_transform(self.y)

    # 定义获得数据集长度的方法
    def __len__(self):
        return len(self.X)

    # 定义获得某一行数据的方法
    def __getitem__(self, idx):
        return [self.X[idx], self.y[idx]]

    # 在类内部定义划分训练集和测试集的方法，在本例中，训练集比例为 0.67，测试集比例为 0.33
    def get_splits(self, n_test=0.33):
        # 确定训练集和测试集的尺寸
        test_size = round(n_test *len(self.X))
        train_size = len(self.X) - test_size
        # 根据尺寸划分训练集和测试集并返回
        #random_split() 函数可用于将数据集拆分为训练集和测试集
        return random_split(self, [train_size, test_size])

测试

In [3]:
# 定义数据集路径（在本例中，数据集需为 csv 文件）
data_path = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/iris.csv'
# 实例化数据集
dataset = CSVDataset(data_path)
print(f'输入矩阵的形状是：{dataset.X.shape}')
# dataset.X  # 查看输入矩阵 dataset.X

输入矩阵的形状是：(150, 4)


In [ ]:
print(f'输出矩阵的形状是：{dataset.y.shape}')
# dataset.y  # 查看输出矩阵

In [ ]:
# len() 方法本质上是调用类内部的 __len__() 方法，所以以下方法是等效的。
print(len(dataset))
print(dataset.__len__())

In [ ]:
# dataset[] 方法本质上是调用类内部的 __getitem__ 方法，所以以下方法是等效的。
print(dataset[149])
print(dataset.__getitem__(149))

加载后，PyTorch 提供 DataLoader 类，用于在模型训练和评估期间导航数据集实例。

可以为训练数据集、测试数据集甚至验证数据集创建 DataLoader 实例。

random_split() 函数可用于将数据集拆分为训练集和测试集。

拆分后，将数据集中 batch 及其 size 提供给 DataLoader，并可选择是否应在每个 epoch 对数据进行随机排序。

In [4]:
from torch.utils.data import DataLoader
from torch.utils.data import random_split

...
# 确定训练集和测试集的尺寸
n_test = 0.33  # 在本例中，训练集比例为 0.67，测试集比例为 0.33
test_size = round(n_test * len(dataset.X))
train_size = len(dataset.X) - test_size

# 根据尺寸划分训练集和测试集并返回
train, test = random_split(dataset, [train_size, test_size])

# 让我们查看一下创建的训练集的类型和长度
print(f'划分的训练集的数据类型是：{type(train)}')
print(f'划分的训练集长度是：{len(train)}')

划分的训练集的数据类型是：<class 'torch.utils.data.dataset.Subset'>
划分的训练集长度是：100


In [ ]:
# list(train)  # 查看一下划分的训练集

# 可以用同样的方法查看划分得到的测试集

可以通过传入数据集中的选定行来定义 DataLoader

In [ ]:
# 为训练集和测试集创建 DataLoader
train_dl = DataLoader(train, batch_size=32, shuffle=True)
test_dl = DataLoader(test, batch_size=1024, shuffle=False)
print(len(train_dl.dataset), len(test_dl.dataset))

循环dataloader，每次迭代生成一批样本

In [ ]:
# 在本例中，train_dl 的 batch_size 为 32，数据将随机排序。让我们来查看一下批次数 train_dl
n_inputs = len(train_dl)
for i, (inputs, targets) in enumerate(train_dl):
    print(f'第 {i} 个 batch 有 {len(inputs)} 个数据，其中输入矩阵的形状是 {inputs.shape}，输出矩阵的形状是 {targets.shape}')
print(f'共有 {n_inputs} 个 batches')

enumerate()函数：用于将一个可遍历的数据对象(如列表、元组或字符串)组合为一个有索引的序列，同时列出数据和数据下标。多用在 for 循环中。

In [5]:
seasons = [('Spring', 'Green'),
           ('Summer', 'Red'),
           ('Fall', 'Yellow'),
           ('Winter', 'White')
           ]
print(list(enumerate(seasons, start=1)))  # start 参数设置序列从 1 开始，不填则默认从 0 开始
print('--------')

# 再在 for 循环中看看 enumerate 函数的效果
for i, (season, color) in enumerate(seasons, start=1):
    print(f'My impression {i} about {season} is {color}.')

[(1, ('Spring', 'Green')), (2, ('Summer', 'Red')), (3, ('Fall', 'Yellow')), (4, ('Winter', 'White'))]
--------
My impression 1 about Spring is Green.
My impression 2 about Summer is Red.
My impression 3 about Fall is Yellow.
My impression 4 about Winter is White.


3、定义模型（单层MLP模型）
在 PyTorch 中定义模型的习惯用法是定义一个继承 Module 类的 Python class 。

你构造的类定义了模型的层，forward() 函数需要覆写以定义在模型层中的输入参数的前向传播。

许多层都可用，例如用于全连接层的 Linear，用于卷积层的 Conv2d，用于池化层的 MaxPool2d。

激活函数也可以定义为层，例如 ReLU, Softmax, 和 Sigmoid.

在构造函数中定义给定层后，也可以初始化给定层的权重。

常见的例子包括 Xavier 和 He weight 权重初始化方案。例如：xavier_uniform_(self.layer.weight)

In [ ]:
from torch.nn import Linear
from torch.nn import ReLU
from torch.nn import Softmax
from torch.nn import Module
from torch.nn.init import kaiming_uniform_
from torch.nn.init import xavier_uniform_

# 定义模型
class MLP(Module):
    # 定义模型属性
    def __init__(self, n_inputs):
        super(MLP, self).__init__()
        # 输出层
        self.hidden1 = Linear(n_inputs, 10)
        #在训练开始前，把hidden1这一层的权重矩阵填上合适的随机初始值
        kaiming_uniform_(self.hidden1.weight, nonlinearity='relu')
        #定义ReLU激活层 ReLU(x) = max(0,x)
        self.act1 = ReLU()
        # 第二个隐藏层
        self.hidden2 = Linear(10, 8)
        kaiming_uniform_(self.hidden2.weight, nonlinearity='relu')
        self.act2 = ReLU()
        # 第三层
        self.hidden3 = Linear(8, 3)
        xavier_uniform_(self.hidden3.weight)
        self.act3 = Softmax(dim=1)

    # 前向传播方法
    def forward(self, X):
        # 输入到第一个隐藏层
        X = self.hidden1(X)
        X = self.act1(X)
        # 第二个隐藏层
        X = self.hidden2(X)
        X = self.act2(X)
        # 输出层
        X = self.hidden3(X)
        X = self.act3(X)
        return X


4、训练模型

训练过程要求定义 损失函数 和 优化算法
常见的损失函数包括：

BCELoss: 用于二元分类的 二元交叉熵损失
CrossEntropyLoss: 用于多元分类的 多元交叉熵损失
MSELoss: 用于回归的 均方损失
使用 随机梯度下降 进行优化，标准算法由 SGD class 提供

In [ ]:
from torch.optim import SGD
from torch.nn import CrossEntropyLoss

...
model = MLP(n_inputs=n_inputs)
# 定义优化器
criterion = CrossEntropyLoss()
optimizer = SGD(model.parameters(), lr=0.01, momentum=0.9)

训练模型涉及枚举训练数据集的 DataLoader。

首先，需要为大量的 training epochs 建立一个循环。然后，需要为每个 mini-batch 建立一个内部循环，用于随机梯度下降。

模型的每次更新都涉及相同的常规模式，包括：

清除最后一个误差梯度。
前向传播并计算模型输出。
计算模型输出的损失。
通过模型反向传播误差。
更新模型以减少损失。

In [ ]:
# 枚举 epochs
for epoch in range(500):
    # 枚举 mini-batches
    for i, (inputs, targets) in enumerate(train_dl):
        # 梯度清除
        optimizer.zero_grad()
        # 计算模型输出
        yhat = model(inputs)
        # 计算损失
        loss = criterion(yhat, targets)
        # 贡献度分配
        loss.backward()
        # 升级模型权重
        optimizer.step()

5、评估模型


可以通过使用测试集的 DataLoader 收集测试集的预测值，然后比较模型预测值与测试集的预期值并计算评价指标

- **`detach()`**：模型输出的 `yhat` 带着计算图（能反传梯度）。评估时不需要梯度，脱离计算图可以**省内存、防止意外反向传播**
- `.numpy()`：PyTorch 张量转 NumPy 数组（前提：在 CPU 上）

In [ ]:
# 纵向拼接数组（把多个批次的预测拼成一个大数组）
from numpy import vstack
# 找最大值的索引（预测的类别）
from numpy import argmax
# 算准确率
from sklearn.metrics import accuracy_score

predictions, actuals = list(), list()  # 实例化预测值列表和预期值列表

for i, (inputs, targets) in enumerate(test_dl):
    # 在测试集上评估模型
    yhat = model(inputs)
    # 转化为 numpy 数据类型
    yhat = yhat.detach().numpy()
    actual = targets.numpy()
    # 转换为类标签
    yhat = argmax(yhat, axis=1)
    # 为 stack reshape 矩阵
    actual = actual.reshape((len(actual), 1))
    yhat = yhat.reshape((len(yhat), 1))
    # 保存数据
    predictions.append(yhat)
    actuals.append(actual)

predictions, actuals = vstack(predictions), vstack(actuals)
# 计算准确度
acc = accuracy_score(actuals, predictions)
print(acc)

6、做出预测

拟合模型可用于对新数据进行预测
预测也将是一个 Tensor

In [ ]:
from torch import Tensor

...
row = [5.1,3.5,1.4,0.2]
# 将数据转化为 Tensor
row = Tensor([row])
# 做出预测
yhat = model(row)
# 重写为 Numpy Array 格式
yhat = yhat.detach().numpy()

print(f'各标签可能的概率： {yhat} (最可能的种类：class={argmax(yhat)})')